In [ ]:
from lets_plot import *

LetsPlot.setup_html()
import polars as pl

In [ ]:
df = pl.read_ipc("benchmarks/results/results.feather")
df = df.with_columns(n_obj=pl.col("n_cells") * pl.col("n_keys"))

group_keys = ["case", "library", "dataset", "n_cells", "n_keys", "key_kind", "replica", "n_obj"]
totals = (
    df.group_by(group_keys)
    .agg(
        pl.col("seconds").sum(),
        pl.col("svg_bytes").max(),
        pl.col("status").first(),
        pl.col("started_at").min(),
        pl.lit("total").alias("phase"),
    )
    .select(df.columns)
)
df = pl.concat([df, totals]).sort(by="library", descending=True)
df.head()

In [ ]:
multi_keys = ["markers", "dotplot", "matrixplot", "heatmap", "highest_expressed"]

In [ ]:
single_keys = ["umap", "violin", "scatter_obs_obs"]

In [ ]:
singles = df.filter(pl.col("case").is_in(single_keys))
multis = df.filter(pl.col("case").is_in(multi_keys))
comp = df.filter(pl.col("case").is_in(multi_keys + single_keys))

In [ ]:
(
    ggplot(multis, aes(x="n_obj", y="seconds", color="library"))
    + geom_point()
    + geom_smooth()
    + facet_grid("case", "phase", x_order=0)
    + theme_bw()
    + scale_y_log10()
    + scale_x_log10(expand=[1, 0.25])
)

In [ ]:
(
    ggplot(singles, aes(x="n_obj", y="seconds", color="library"))
    + geom_point()
    + geom_smooth()
    + facet_grid("case", "phase", x_order=0)
    + theme_bw()
    + scale_y_log10()
    + scale_x_log10(expand=[1, 0.25])
)

In [ ]:
(
    ggplot(comp, aes(x="n_obj", y="seconds", color="library"))
    + geom_point()
    + geom_smooth()
    + facet_grid("case", "phase", x_order=0)
    + theme_bw()
    + scale_y_log10()

    + scale_x_log10(expand=[1, 0.25])
)